# Hyperliquid Portfolio & PnL Dashboard

Tracks account balances, open perp positions, and realized/unrealized PnL including trading fees and signed perp funding from Hyperliquid data cached in CSV. Calculates portfolio performance and risk metrics (including VaR and historical volatility), attributes PnL by asset, and compares account performance with BTC through charts and summary statistics.


In [1]:
import json
from pathlib import Path

# Enter your wallet address here to fetch data for that specific wallet 
WALLET_ADDRESS = "0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248"
#WALLET_ADDRESS = "0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2"


implementation_path = Path("hyperliquid_account_analytics_implementation.ipynb")
implementation_notebook = json.loads(implementation_path.read_text(encoding="utf-8"))
implementation_namespace = {"__name__": "hyperliquid_analytics"}
for cell in implementation_notebook["cells"]:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        if source.strip():
            exec(compile(source, f"{implementation_path} cell", "exec"), implementation_namespace)

HyperliquidAnalytics = implementation_namespace["HyperliquidAnalytics"]
analytics = HyperliquidAnalytics(WALLET_ADDRESS)
_ = analytics.fetch_data()

## Account / PnL

Interactive account and risk summary, including account and portfolio values.

**Account value** is the latest daily all-time account-value point in the local history cache. **Portfolio value** is fetched live from Hyperliquid's portfolio endpoint. They are the same API-reported account-value measure at different freshness points, so they can differ if the cached history is behind. Neither value is the open-position notional.

In [2]:
analytics.show_account_pnl()

=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,$790.97
Portfolio value,$785.43
Account 1D,12.86%
Realized 1D,$0.01
Realized 1D (% of account value),0.00%
Unrealized PnL,$-12.33
Unrealized PnL (% of account value),-1.56%
Unrealized 1D change,$12.24
Average realized PnL / day,$-0.63
Average unrealized PnL change / day,$12.24


## Account charts

Interactive Plotly charts show account value, daily realized and unrealized PnL, cumulative PnL, and weekly return distribution.

In [3]:
analytics.show_account_charts()

## Account vs BTC

Shows the intraday live-snapshot index and monthly lifetime realized-PnL reconstruction.

In [4]:
analytics.show_account_vs_btc()

Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.


## Traded volume vs realized PnL chart

Daily fill volume compared with realized PnL by market type.

In [5]:
analytics.show_volume_chart()

## Open positions

Interactive sortable/filterable table of perp positions, including margin type and current plus annualized funding rates.

In [ ]:
analytics.show_open_positions()

## Realized PnL statistics

Interactive tables report fee-net spot/perp results with signed perp funding, wins, losses, and totals. The perp output also includes liquidation count, share of closes, and liquidation PnL.

In [7]:
analytics.show_realized_pnl()

=== REALIZED SPOT ===


Metric,Value
Realized PnL,$0.00 (0.00%)
Funding,n/a
1D,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$0.00 (0.00%)
QTD,$0.00 (0.00%)
YTD,$0.00 (0.00%)
W,0
Win %,n/a
Avg W,n/a


=== REALIZED PERP ===


Metric,Value
Realized PnL,$-5.64 (-0.73%)
Funding,$-1.20 (-0.15%)
1D,$-2.23 (-0.29%)
7D,$-5.78 (-0.75%)
30D,$-5.64 (-0.73%)
QTD,$-5.64 (-0.73%)
YTD,$-5.64 (-0.73%)
W,8
Win %,66.67%
Avg W,$1.02



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$-2.27,-40.27%
Fees,$-2.17,-38.52%
Funding,$-1.20,-21.21%
Net perp PnL,$-5.64,


Trading is gross of fees; fees are shown as a negative contribution. Percentages use the absolute net PnL as denominator, so their signs match the dollar contributions.
Liquidations: 0 of 12             closes (0.00%) | liquidation PnL: $0.00 (-0.00% of perp PnL)
Realized PnL uses fill closedPnl (trading fees included) plus signed perp funding; funding paid is negative.
Fill PnL, fees, win/loss, and liquidation stats use available fills (API cap: 10,000); funding history is stored separately.


## Traded volume vs PnL

Interactive table compares turnover efficiency and correlations by market type. Realized PnL / Volume (%) is realized fill PnL divided by one-sided traded notional; funding is excluded.

In [8]:
analytics.show_volume_summary()

=== TRADED VOLUME VS REALIZED PNL ===


Market,Volume ($),Realized PnL ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$4,896.78",$-4.44,-0.09%,$-0.91,6,-0.705,-0.725


Volume is one-sided fill notional (price ? size), counted once per fill. Realized PnL is Hyperliquid closedPnl, which includes trading fees; funding is excluded.
Correlation describes co-movement, not causation. Daily observations with no fills are omitted.


## PnL attribution

Interactive highlighted table breaks realized PnL, open perp uPnL, and total PnL out by asset. The Closes column shows distinct perp closes per asset, counted from available fills. Each percentage column is the signed contribution as a share of that column's gross absolute PnL, so gains are positive, losses are negative, and the colors match the dollar values. Percentages can sum to less than 100% when gains and losses offset.


In [9]:
analytics.show_pnl_attribution()

=== PNL ATTRIBUTION ===


Type,Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
Perp,para:SOFI,1,$-10.08,-61.80%,$0.00,0.00%,$-10.08,-19.84%
Perp,PONS,0,$0.10,0.64%,$-7.67,-21.86%,$-7.56,-14.89%
Perp,INIT,0,$-0.09,-0.52%,$-5.50,-15.69%,$-5.59,-11.00%
Perp,FOGO,0,$0.04,0.22%,$5.05,14.40%,$5.09,10.01%
Perp,GRASS,0,$0.01,0.09%,$4.45,12.69%,$4.47,8.79%
Perp,WIF,0,$0.06,0.38%,$-3.08,-8.78%,$-3.02,-5.94%
Perp,BLUR,2,$2.34,14.32%,$0.00,0.00%,$2.34,4.60%
Perp,XPL,0,$0.01,0.06%,$2.09,5.94%,$2.09,4.12%
Perp,REZ,1,$2.02,12.39%,$0.00,0.00%,$2.02,3.98%
Perp,XTEN,0,$0.05,0.30%,$-1.75,-4.98%,$-1.70,-3.34%


Net attributed PnL: $-17.55 | percentages are signed             shares of gross absolute PnL in each column, so their signs match the dollar                 contributions.
Scope: available realized fills (API cap: 10,000) + saved perp funding + current open perp uPnL; spot unrealized PnL is not included.
